# Preprocessing & Training Deteksi Bahasa Isyarat (BISINDO) dengan YOLOv11

**Skripsi:** Deteksi Bahasa Isyarat Indonesia Menggunakan Deep Learning
**Penyusun:** Abil — Teknik Informatika, Universitas Gunadarma

Notebook ini digunakan untuk dataset yang **sudah di-split** menjadi 3 folder terpisah: `train`, `valid`, `test`,
dan sudah dikemas dalam **satu file ZIP**. Masing-masing folder di dalam ZIP berisi gambar beserta file
`_annotations.csv` dengan format:

```
filename,width,height,class,xmin,ymin,xmax,ymax
```

Alur notebook ini:
1. Upload file ZIP dataset (cukup satu file)
2. Ekstraksi ZIP & deteksi otomatis folder train/valid/test
3. Membaca file `_annotations.csv` dari masing-masing folder
4. Membuat mapping kelas (string → index) yang **konsisten** di seluruh split
5. Mengonversi anotasi Pascal VOC (xmin, ymin, xmax, ymax) ke format YOLO ternormalisasi
6. Menyusun ulang struktur folder menjadi format standar YOLO (`images/` & `labels/`)
7. Membuat file `data.yaml`
8. **Melatih model deteksi objek menggunakan YOLOv11 (Ultralytics)**
9. Evaluasi model pada data test & contoh inferensi

> **Catatan:** Kamu hanya perlu upload file ZIP dataset di sel Bagian 2 — tidak perlu mengatur path folder secara manual (kecuali deteksi otomatis gagal).


## 1. Import Library

In [ ]:
import os
import shutil
import random
from pathlib import Path

import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

# Untuk reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Semua library berhasil diimpor.")


## 2. Upload Dataset (ZIP)

Cukup upload **satu file ZIP** yang di dalamnya sudah berisi 3 folder split: `train`, `valid`, `test`
(nama folder boleh variasinya seperti `val`/`validation`, `training`, `testing`, dsb — akan dideteksi
otomatis). Setiap folder split harus berisi gambar + file `_annotations.csv`.

Contoh isi ZIP:
```
dataset.zip
├── train/
│   ├── _annotations.csv
│   ├── photo1.jpg
│   └── ...
├── valid/
│   ├── _annotations.csv
│   └── ...
└── test/
    ├── _annotations.csv
    └── ...
```

- Jika dijalankan di **Google Colab**, sel di bawah akan menampilkan tombol upload interaktif.
- Jika dijalankan di **Jupyter lokal**, isi `ZIP_PATH` secara manual ke lokasi file ZIP kamu.


In [ ]:
import zipfile

# Nama file anotasi di setiap folder split
ANNOTATION_FILENAME = "_annotations.csv"

# Folder tempat hasil ekstraksi ZIP disimpan
EXTRACT_DIR = "dataset_extracted"

# Folder output dataset hasil preprocessing (format YOLO: images/ & labels/)
OUTPUT_DIR = "dataset_yolo"

splits = ["train", "valid", "test"]

# Deteksi apakah sedang berjalan di Google Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Silakan upload file ZIP dataset kamu (berisi folder train/valid/test)...")
    uploaded = colab_files.upload()
    ZIP_PATH = list(uploaded.keys())[0]
else:
    # ==============================
    # JIKA BUKAN DI GOOGLE COLAB — ISI PATH ZIP SECARA MANUAL DI SINI
    # ==============================
    ZIP_PATH = "dataset.zip"  # <-- ganti sesuai nama/lokasi file ZIP kamu

print(f"\nFile ZIP yang akan digunakan: {ZIP_PATH}")


## 3. Ekstraksi ZIP & Deteksi Otomatis Folder Train/Valid/Test

Sel di bawah akan mengekstrak ZIP, lalu mencari folder `train`, `valid`, dan `test` (termasuk variasi
namanya) yang memiliki file `_annotations.csv` di dalamnya — walaupun ZIP berisi satu folder pembungkus
tambahan di root (mis. `dataset.zip/dataset/train/...`).

In [ ]:
# Bersihkan folder ekstraksi sebelumnya (jika ada), lalu ekstrak ZIP
if Path(EXTRACT_DIR).exists():
    shutil.rmtree(EXTRACT_DIR)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print(f"Dataset berhasil diekstrak ke: {EXTRACT_DIR}\n")

# Tampilkan struktur folder hasil ekstraksi (maksimal 2 level ke bawah)
print("Struktur folder hasil ekstraksi:")
for p in sorted(Path(EXTRACT_DIR).rglob("*")):
    depth = len(p.relative_to(EXTRACT_DIR).parts)
    if depth <= 2:
        print("  " * depth + p.name + ("/" if p.is_dir() else ""))


In [ ]:
def find_split_dirs(root_dir, ann_filename=ANNOTATION_FILENAME):
    '''Mencari folder train/valid/test (beserta variasi namanya) di dalam root_dir,
    dengan syarat folder tersebut memiliki file anotasi ann_filename di dalamnya.

    Returns
    -------
    dict
        Mapping {'train': path, 'valid': path, 'test': path} untuk folder yang berhasil ditemukan
    '''
    root_dir = Path(root_dir)
    aliases = {
        "train": ["train", "training"],
        "valid": ["valid", "val", "validation"],
        "test": ["test", "testing"],
    }

    all_dirs = [p for p in root_dir.rglob("*") if p.is_dir()]
    all_dirs.append(root_dir)

    found = {}
    for split, names in aliases.items():
        for d in all_dirs:
            if d.name.lower() in names and (d / ann_filename).exists():
                found[split] = str(d)
                break

    return found


SOURCE_DIRS = find_split_dirs(EXTRACT_DIR)

missing_splits = [s for s in splits if s not in SOURCE_DIRS]

if missing_splits:
    print(f"[Peringatan] Folder split berikut TIDAK berhasil dideteksi otomatis: {missing_splits}")
    print("Silakan periksa struktur ZIP kamu, atau isi SOURCE_DIRS secara manual, contoh:")
    print('  SOURCE_DIRS["train"] = "dataset_extracted/nama_folder_train"')
else:
    print("Folder split berhasil dideteksi otomatis:\n")
    for split_name, path in SOURCE_DIRS.items():
        csv_path = Path(path) / ANNOTATION_FILENAME
        print(f"  [{split_name:5s}] {path}  ({csv_path.name}: {'DITEMUKAN' if csv_path.exists() else 'TIDAK DITEMUKAN'})")


## 4. Membaca dan Memeriksa Dataset Anotasi per Split

Kolom yang diharapkan: `filename`, `width`, `height`, `class`, `xmin`, `ymin`, `xmax`, `ymax`

In [ ]:
dfs = {}

for split_name, src in SOURCE_DIRS.items():
    csv_path = Path(src) / ANNOTATION_FILENAME
    df_split = pd.read_csv(csv_path)
    dfs[split_name] = df_split

    print("=" * 55)
    print(f"SPLIT: {split_name.upper()}")
    print("=" * 55)
    print(f"Total baris anotasi : {len(df_split)}")
    print(f"Jumlah gambar unik  : {df_split['filename'].nunique()}")
    print(f"Jumlah kelas unik   : {df_split['class'].nunique()}")
    print()

dfs["train"].head()


In [ ]:
# Cek missing values & tipe data numerik pada setiap split
required_numeric_cols = ["width", "height", "xmin", "ymin", "xmax", "ymax"]

for split_name, df_split in dfs.items():
    missing = df_split.isnull().sum()
    print(f"[{split_name.upper()}]")
    if missing.sum() > 0:
        print("  Peringatan: Ditemukan missing values!")
        print(missing[missing > 0])
    else:
        print("  Tidak ada missing values.")
    print(f"  Tipe data numerik OK: {all(pd.api.types.is_numeric_dtype(df_split[c]) for c in required_numeric_cols)}")
    print()


In [ ]:
# Distribusi jumlah anotasi per kelas, untuk masing-masing split
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, split_name in zip(axes, splits):
    dist = dfs[split_name]['class'].value_counts().sort_index()
    dist.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Distribusi Kelas - {split_name.upper()}')
    ax.set_xlabel('Kelas')
    ax.set_ylabel('Jumlah Anotasi')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 5. Mapping Kelas (Label String → Index)

Mapping kelas dibuat dari **gabungan** kelas di seluruh split (train + valid + test) agar index kelas
konsisten di semua bagian dataset, meskipun ada kelas yang kebetulan tidak muncul di salah satu split.

In [ ]:
# Gabungkan semua kelas dari ketiga split
all_classes = set()
for df_split in dfs.values():
    all_classes.update(df_split['class'].unique().tolist())

class_names = sorted(all_classes)
class_to_index = {name: idx for idx, name in enumerate(class_names)}
index_to_class = {idx: name for name, idx in class_to_index.items()}

print(f"Jumlah kelas total: {len(class_names)}")
print("\nMapping kelas -> index:")
for name, idx in class_to_index.items():
    print(f"  {name:10s} -> {idx}")

# Cek konsistensi: apakah ada kelas yang hanya muncul di salah satu split?
for split_name, df_split in dfs.items():
    missing_here = all_classes - set(df_split['class'].unique().tolist())
    if missing_here:
        print(f"\n[Peringatan] Split '{split_name}' tidak memiliki contoh untuk kelas: {sorted(missing_here)}")


## 6. Fungsi Konversi Bounding Box (Pascal VOC → YOLO)

Format YOLO per baris label: `class_index x_center y_center width height` (semua nilai ternormalisasi 0-1).

In [ ]:
def convert_bbox_to_yolo(row):
    '''Mengonversi satu baris anotasi (Pascal VOC) ke satu baris format YOLO.

    Parameters
    ----------
    row : pandas.Series
        Baris anotasi dengan kolom: width, height, class, xmin, ymin, xmax, ymax

    Returns
    -------
    str
        Baris teks format YOLO: 'class_index x_center y_center width height'
    '''
    img_w, img_h = row['width'], row['height']
    xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']

    # Pastikan koordinat berada dalam batas gambar
    xmin = max(0, min(xmin, img_w))
    xmax = max(0, min(xmax, img_w))
    ymin = max(0, min(ymin, img_h))
    ymax = max(0, min(ymax, img_h))

    box_w = xmax - xmin
    box_h = ymax - ymin
    x_center = xmin + (box_w / 2)
    y_center = ymin + (box_h / 2)

    # Normalisasi ke rentang 0-1
    x_center_norm = x_center / img_w
    y_center_norm = y_center / img_h
    w_norm = box_w / img_w
    h_norm = box_h / img_h

    class_idx = class_to_index[row['class']]

    return f"{class_idx} {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}"


## 7. Menyusun Ulang Struktur Folder ke Format YOLO

Untuk setiap split (train/valid/test), gambar akan disalin ke `OUTPUT_DIR/<split>/images/` dan
label `.txt` hasil konversi akan disimpan di `OUTPUT_DIR/<split>/labels/`.

In [ ]:
def process_split(split_name, df_annotations, images_source_dir, output_dir):
    '''Menyalin gambar & membuat label YOLO untuk satu split.

    Parameters
    ----------
    split_name : str
        'train', 'valid', atau 'test'
    df_annotations : pandas.DataFrame
        Dataframe anotasi untuk split ini (dari _annotations.csv)
    images_source_dir : str
        Folder sumber gambar untuk split ini
    output_dir : str
        Folder output dataset (root)

    Returns
    -------
    dict
        Ringkasan hasil proses
    '''
    images_dest = Path(output_dir) / split_name / "images"
    labels_dest = Path(output_dir) / split_name / "labels"
    images_dest.mkdir(parents=True, exist_ok=True)
    labels_dest.mkdir(parents=True, exist_ok=True)

    success_count = 0
    missing_images = []
    total_annotations = 0

    filenames = df_annotations['filename'].unique()

    for filename in filenames:
        source_image_path = Path(images_source_dir) / filename
        dest_image_path = images_dest / filename

        if not source_image_path.exists():
            missing_images.append(filename)
            continue

        shutil.copy2(source_image_path, dest_image_path)
        success_count += 1

        image_annotations = df_annotations[df_annotations['filename'] == filename]

        yolo_lines = []
        for _, row in image_annotations.iterrows():
            yolo_lines.append(convert_bbox_to_yolo(row))
            total_annotations += 1

        label_filename = Path(filename).stem + ".txt"
        label_path = labels_dest / label_filename

        with open(label_path, "w") as f:
            f.write("\n".join(yolo_lines))

    return {
        "split": split_name,
        "total_files": len(filenames),
        "success": success_count,
        "missing": len(missing_images),
        "missing_files": missing_images,
        "total_annotations": total_annotations,
    }


In [ ]:
# Jalankan proses untuk masing-masing split
results = []

for split_name in splits:
    r = process_split(split_name, dfs[split_name], SOURCE_DIRS[split_name], OUTPUT_DIR)
    results.append(r)

print("=" * 55)
print("RINGKASAN HASIL PREPROCESSING")
print("=" * 55)
for r in results:
    print(f"\nSplit: {r['split'].upper()}")
    print(f"  Total file target       : {r['total_files']}")
    print(f"  Berhasil disalin        : {r['success']}")
    print(f"  Gambar tidak ditemukan  : {r['missing']}")
    print(f"  Total anotasi (bbox)    : {r['total_annotations']}")
    if r['missing'] > 0:
        print(f"  Contoh file hilang      : {r['missing_files'][:5]}")

print("\n" + "=" * 55)
total_success = sum(r['success'] for r in results)
total_target = sum(r['total_files'] for r in results)
print(f"Total gambar berhasil diproses: {total_success}/{total_target}")


## 8. Verifikasi Hasil Preprocessing

In [ ]:
print("Verifikasi jumlah file per folder:\n")

for split_name in splits:
    images_path = Path(OUTPUT_DIR) / split_name / "images"
    labels_path = Path(OUTPUT_DIR) / split_name / "labels"

    num_images = len(list(images_path.glob("*")))
    num_labels = len(list(labels_path.glob("*.txt")))

    status = "OK" if num_images == num_labels else "TIDAK COCOK — periksa kembali!"

    print(f"[{split_name.upper()}]")
    print(f"  Jumlah gambar : {num_images}")
    print(f"  Jumlah label  : {num_labels}")
    print(f"  Status        : {status}\n")


In [ ]:
# Tampilkan contoh isi salah satu file label hasil konversi
sample_label_files = list((Path(OUTPUT_DIR) / "train" / "labels").glob("*.txt"))

if sample_label_files:
    sample_file = sample_label_files[0]
    print(f"Contoh isi file label: {sample_file.name}\n")
    with open(sample_file, "r") as f:
        print(f.read())
else:
    print("Tidak ditemukan file label untuk ditampilkan.")


## 9. Membuat File Konfigurasi `data.yaml`

File ini digunakan sebagai konfigurasi saat training model YOLOv11 (Ultralytics).

In [ ]:
data_yaml_content = {
    "train": str((Path(OUTPUT_DIR) / "train" / "images").resolve()),
    "val": str((Path(OUTPUT_DIR) / "valid" / "images").resolve()),
    "test": str((Path(OUTPUT_DIR) / "test" / "images").resolve()),
    "nc": len(class_names),
    "names": class_names,
}

yaml_path = Path(OUTPUT_DIR) / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"File data.yaml berhasil dibuat di: {yaml_path}\n")
print("Isi file data.yaml:")
print("-" * 40)
with open(yaml_path, "r") as f:
    print(f.read())


## 10. Ringkasan Dataset

In [ ]:
print("=" * 55)
print("RINGKASAN AKHIR PREPROCESSING DATASET")
print("=" * 55)
print(f"Total kelas          : {len(class_names)}")
print(f"Daftar kelas         : {class_names}")
print(f"Total gambar Train   : {len(list((Path(OUTPUT_DIR)/'train'/'images').glob('*')))}")
print(f"Total gambar Valid   : {len(list((Path(OUTPUT_DIR)/'valid'/'images').glob('*')))}")
print(f"Total gambar Test    : {len(list((Path(OUTPUT_DIR)/'test'/'images').glob('*')))}")
print(f"Lokasi dataset       : {Path(OUTPUT_DIR).resolve()}")
print(f"File konfigurasi     : {yaml_path.resolve()}")
print("=" * 55)


## 11. Instalasi Ultralytics (YOLOv11)

Jalankan sel ini sekali untuk menginstal library `ultralytics` yang mendukung YOLOv11.

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()


## 12. Training Model YOLOv11

Beberapa pilihan model pretrained YOLOv11 dari Ultralytics (semakin besar, semakin akurat namun lebih lambat):
- `yolo11n.pt` — Nano (tercepat, cocok untuk perangkat terbatas)
- `yolo11s.pt` — Small
- `yolo11m.pt` — Medium
- `yolo11l.pt` — Large
- `yolo11x.pt` — Extra Large (paling akurat, paling berat)

Untuk skripsi/eksperimen awal, `yolo11n.pt` atau `yolo11s.pt` biasanya menjadi titik awal yang baik.
Sesuaikan `EPOCHS`, `IMG_SIZE`, dan `BATCH_SIZE` dengan kapasitas GPU yang tersedia.

In [ ]:
from ultralytics import YOLO

# ==============================
# KONFIGURASI TRAINING — SESUAIKAN INI
# ==============================
MODEL_ARCH = "yolo11n.pt"   # pilihan: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16
PROJECT_NAME = "runs_bisindo"
RUN_NAME = "yolo11_bisindo"

# Muat model pretrained (transfer learning dari COCO)
model = YOLO(MODEL_ARCH)

# Mulai training
train_results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    seed=RANDOM_SEED,
    project=PROJECT_NAME,
    name=RUN_NAME,
    patience=20,        # early stopping jika tidak ada peningkatan selama 20 epoch
    plots=True,
)


## 13. Evaluasi Model pada Data Test

In [ ]:
# Evaluasi menggunakan bobot terbaik hasil training
best_weights_path = Path(PROJECT_NAME) / RUN_NAME / "weights" / "best.pt"
print(f"Menggunakan bobot terbaik: {best_weights_path}")

best_model = YOLO(str(best_weights_path))

metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=IMG_SIZE,
)

print("\nRingkasan metrik pada data test:")
print(f"  mAP50    : {metrics.box.map50:.4f}")
print(f"  mAP50-95 : {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall   : {metrics.box.mr:.4f}")


## 14. Visualisasi Hasil Training

Ultralytics otomatis menyimpan grafik loss/mAP (`results.png`) dan confusion matrix di folder run.

In [ ]:
run_dir = Path(PROJECT_NAME) / RUN_NAME

results_png = run_dir / "results.png"
confusion_matrix_png = run_dir / "confusion_matrix.png"

for img_path, title in [(results_png, "Kurva Training (Loss & mAP)"),
                          (confusion_matrix_png, "Confusion Matrix")]:
    if img_path.exists():
        img = plt.imread(img_path)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.show()
    else:
        print(f"File tidak ditemukan: {img_path}")


## 15. Contoh Inferensi pada Gambar Test

In [ ]:
# Ambil beberapa contoh gambar dari folder test untuk inferensi
test_images_dir = Path(OUTPUT_DIR) / "test" / "images"
sample_images = list(test_images_dir.glob("*"))[:5]

if sample_images:
    predict_results = best_model.predict(
        source=[str(p) for p in sample_images],
        imgsz=IMG_SIZE,
        conf=0.25,
        save=True,
        project=PROJECT_NAME,
        name=f"{RUN_NAME}_predict",
    )

    for res in predict_results:
        plt.figure(figsize=(8, 8))
        plt.imshow(res.plot()[:, :, ::-1])  # BGR -> RGB
        plt.axis('off')
        plt.title(Path(res.path).name)
        plt.show()
else:
    print("Tidak ditemukan gambar pada folder test untuk inferensi.")


## 16. Export Model (Opsional)

Model dapat diekspor ke format lain (mis. ONNX) untuk keperluan deployment, misalnya jika ingin
digunakan pada aplikasi web/mobile di luar Python.

In [ ]:
# Uncomment untuk export ke ONNX
# export_path = best_model.export(format="onnx", imgsz=IMG_SIZE)
# print(f"Model berhasil diekspor ke: {export_path}")

print("Lewati/aktifkan sel ini sesuai kebutuhan deployment.")
